# Example: CCD++ on 100k, using the shared infrastructure


1. the same 100k split (`ua.base` / `ua.test`), and
2. the same results file and columns (`als/src/results_utils.py`).

To compare fairly we use the **same dataset (100k), same hyperparameters, and
the same number of runs** on both sides.

For the bigger datasets the idea is the same but you read the Parquet
split saved under `data/processed/<dataset>/` instead of the 100k files.

In [ ]:
import sys
from pathlib import Path
from time import perf_counter


PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "als").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "ccdpp" / "src"))   # your code
sys.path.append(str(PROJECT_ROOT / "als" / "src"))     # shared infrastructure

# your code
from data_utils import load_ml100k_split
from ccdpp import train_ccdpp_fixed, predict, rmse
# shared infrastructure
from datasets import ensure_movielens
from results_utils import make_result, append_result, new_run_id

In [ ]:

DATASET  = "ml-100k"
K        = 5
LAMBDA   = 0.1
MAX_ITER = 10
SEED     = 42
NUM_RUNS = 3       
RESULTS_CSV = str(PROJECT_ROOT / "results" / "als_results.csv")

## 1. Load the data
`ensure_movielens` downloads 100k the first time. `load_ml100k_split` is your
loader and gives back the sparse training matrix and the test arrays.

In [3]:
data_dir = ensure_movielens(DATASET, PROJECT_ROOT / "data")
data = load_ml100k_split(str(data_dir))

R_train = data["R_train"]   # scipy sparse training matrix
u_test  = data["u_test"]    # test user indices
i_test  = data["i_test"]    # test item indices
y_true  = data["y_true"]    # test ratings

train_size = R_train.nnz
test_size  = len(y_true)
print("train:", train_size, "| test:", test_size)

train: 90570 | test: 9428


## 2. Run CCD++ NUM_RUNS times and log each run
Each run trains (timed), predicts, computes RMSE, and appends one row in the
shared schema with `algorithm="CCD++"`. `num_cores=1` because this serial
version is single-threaded.

In [4]:
for run in range(1, NUM_RUNS + 1):
    start = perf_counter()
    W, H = train_ccdpp_fixed(R_train, k=K, lam=LAMBDA, n_iter=MAX_ITER, seed=SEED)
    train_time = perf_counter() - start

    start = perf_counter()
    y_pred = predict(W, H, u_test, i_test)
    test_rmse = rmse(y_true, y_pred)
    eval_time = perf_counter() - start

    row = make_result(
        algorithm="CCD++",
        dataset=DATASET,
        dataset_size=train_size + test_size,
        num_cores=1,
        k=K,
        lam=LAMBDA,
        seed=SEED,
        max_iter=MAX_ITER,
        train_size=train_size,
        test_size=test_size,
        train_time=train_time,
        test_rmse=test_rmse,
        eval_time=eval_time,
        run_id=new_run_id(),
    )
    append_result(row, RESULTS_CSV)
    print(f"  run {run}/{NUM_RUNS}: RMSE={test_rmse:.4f}  train_time={train_time:.2f}s")

  run 1/3: RMSE=1.0987  train_time=0.11s


  run 2/3: RMSE=1.0987  train_time=0.10s


  run 3/3: RMSE=1.0987  train_time=0.11s


## 3. Compare ALS vs CCD++ (same 100k, same k/lambda/maxIter, same runs)
We keep only the rows with the same settings, then take the last `NUM_RUNS` of
each algorithm, so it is a fair side-by-side.

In [5]:
import pandas as pd
df = pd.read_csv(RESULTS_CSV)

same = df[
    (df.dataset == DATASET)
    & (df.k == K)
    & (df["lambda"] == LAMBDA)
    & (df.max_iter == MAX_ITER)
]
same = same.groupby("algorithm").tail(NUM_RUNS)

comparison = (
    same.groupby("algorithm")
    .agg(
        runs=("run_id", "count"),
        mean_rmse=("test_rmse", "mean"),
        mean_train_time=("train_time", "mean"),
        mean_total_time=("total_time", "mean"),
    )
    .round(4)
)
comparison

,runs,mean_rmse,mean_train_time,mean_total_time
algorithm,,,,
ALS,3,0.9483,0.7356,0.8911
CCD++,3,1.0987,0.1078,0.1085
